Run this notebook to

- Apply a high-pass filter to 8Hz
- Create stimulus epochs (300ms before to 1s after stimulus, baselined at 100ms before to time of stimulus)
- Drop epochs that fall within EEG parts annotated as bad
- Set behavioural meta-data
- Drop epochs that exceed peak-to-peak amplitude range of 200 µV 
- Sub-select task block data (ie not practice trials)
- Plot epoch timecourse for all channels
- Plot electrode Pz trace with stacked ERP plot sorted by reaction times
- Plot electrode Pz trace for correct vs error trials
- Calculate z-scored confidence reports within each partner condition and plot Pz trace for above vs below median confidence trials



In [6]:
import os
import numpy as np
import mne
from mne.preprocessing import (ICA, create_eog_epochs, create_ecg_epochs,corrmap)
import matplotlib.pyplot as plt
import seaborn as sns
import re
import pandas as pd

million = 1000000.


%matplotlib qt

input_dir = 'EEGdata/cleaned'



def mkdir(p):
    sp = re.split('/|\\\\', p)
    bp = ''
    for pp in sp:
        bp = os.path.join(bp, pp)
        if not os.path.exists(bp):
            os.mkdir(bp)
            print( '%s created.' % bp)

            
mkdir('TaskstimulusEpochs')
mkdir('Figures/TaskstimulusEpochs')
mkdir('Figures/TaskstimulusEpochs/all channels')
mkdir('Figures/TaskstimulusEpochs/correct vs error')
mkdir('Figures/TaskstimulusEpochs/z-scored confidence median split')
mkdir('Figures/TaskstimulusEpochs/ERP by reaction time')



def plot_joint(erp, times, title='', width=12, height=8, invert=True, save=None):
    fig = erp.plot_joint(times=times,
                         show=False,
                         ts_args=dict(time_unit='s'),
                         topomap_args=dict(res=128, contours=4, time_unit='s'),
                         title=title)
    fig.set_figwidth(width)
    fig.set_figheight(height)
    axes = fig.get_axes()
    ax0 = axes[0]
    if invert:
        ax0.invert_yaxis()
    ch = ax0.get_children()
    for c in ch:
        if type(c) == plt.Annotation:
            c.remove()
        if type(c) == plt.Line2D:
            c.set_linewidth(2.5)
            c.set_alpha(.75)
    leg_ax = axes[-2]
    leg_ax.get_children()[0].set_sizes([50.])
    leg_ax.set_aspect('equal')
    if save is not None:
        fig.savefig(save)
    fig.show()



sessions = [1, 2]

## CHECK PARTICIPANTS 5 and 8! (and 16 for dropping loads of epochs)

participant_numbers = [1, 2, 3, 4,  6, 7, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20, 21]


for session in sessions:
    for subject in participant_numbers:
        print('>>>> Session %i' % session)
        print('>>>> Participant %i' % subject)
        
        if session == 1:
            raw = mne.io.read_raw_fif('%s/s%i-raw.fif' % (input_dir, subject)).load_data() # session 1
            annot_file = '%s/%i-annot.fif' % ('annotations', subject)

        if session == 2:
            raw = mne.io.read_raw_fif('%s/s%i_2-raw.fif' % (input_dir, subject)).load_data() # session 2
            annot_file = '%s/%i_2-annot.fif' % ('annotations', subject)


        annotations = mne.read_annotations(annot_file)
        raw = raw.set_annotations(annotations)
        
        raw = raw.filter(l_freq=None, h_freq=8)
        
        events = mne.find_events(raw, stim_channel='Status')
        stimulus_event_dict = {'stimulus_left_correct': 4,
                               'stimulus_right_correct': 5
                              }
        stimulus_epochs = mne.Epochs(raw, events, tmin=-0.3, tmax=1, event_id=stimulus_event_dict,
                                     reject_by_annotation=True, preload=True, baseline=(-0.1, 0))
        
        drop_log = [v[0]  if (len(v) > 0) else 'ok' for v in stimulus_epochs.drop_log]
        drop_log = np.array(drop_log)
        drop_log = drop_log[drop_log != 'IGNORED']
        
        if len(stimulus_epochs) != len(drop_log[drop_log != 'BAD_']):
            raise Exception('Numbers of epochs do not match')
        
        
        alldata = pd.read_csv("mergedData/allData.csv")
        data = alldata[alldata['participant']== subject]
        
        if session == 1:
            data = data[data['session']== 1] # session 1
        if session == 2:
            data = data[data['session']== 2] # session 2
            
        data['droplog'] = drop_log
        data = data[data['droplog'] == 'ok']
        data.index = range(len(data))
        
        if len(stimulus_epochs) != len(data):
            raise Exception('Number of epochs do not match.\n Found %i EEG events and %i csv events' % (len(epochs['stimulus']), len(data)))
        
        metadata = pd.DataFrame(data=data)
        stimulus_epochs.metadata = metadata
        
        reject_criteria = dict(eeg=200e-6)
        stimulus_epochs.drop_bad(reject=reject_criteria)
        
        task_stimulus_epochs = stimulus_epochs['block_count > 2']
        
        if session == 1:
            task_stimulus_epochs.save('TaskstimulusEpochs/%i_1-epo.fif' % (subject), overwrite=True) # session 1
        if session == 2:
            task_stimulus_epochs.save('TaskstimulusEpochs/%i_2-epo.fif' % (subject), overwrite=True) # session 2
            
        condition = task_stimulus_epochs.metadata['condition'].unique()
        condition = condition[0]
        
        
        erp = task_stimulus_epochs.copy().average()
        plot_joint(erp, [-0.3, 0, 0.3, 0.6], title='Participant %i - stimulus-locked - %s' % (subject, condition));
        if session == 1:
            plt.savefig('Figures/TaskstimulusEpochs/all channels/%i_1.png' % (subject)) # session 1
        if session == 2:
            plt.savefig('Figures/TaskstimulusEpochs/all channels/%i_2.png' % (subject)) # session 2
            
            
        sort_order = np.argsort(task_stimulus_epochs.metadata['decision_rt'])
        task_stimulus_epochs.plot_image(order=sort_order, picks='Pz') # set sigma to 1 or so if you want temporal smoothing
        if session == 1:
            plt.savefig('Figures/TaskstimulusEpochs/ERP by reaction time/%i_1.png' % (subject)) # session 1
        if session == 2:
            plt.savefig('Figures/TaskstimulusEpochs/ERP by reaction time/%i_2.png' % (subject)) # session 2
            
            
        correct_epochs = task_stimulus_epochs['participant_correct == True']
        correct_evoked = correct_epochs.average()
        incorrect_epochs = task_stimulus_epochs['participant_correct == False']
        incorrect_evoked = incorrect_epochs.average()
        evokeds = dict(correct=correct_evoked,incorrect=incorrect_evoked)
        mne.viz.plot_compare_evokeds(evokeds, picks=['Pz'], invert_y=False)
        plt.title('stimulus-locked | P%i | %s' % (subject, condition))
        if session == 1:
            plt.savefig('Figures/TaskstimulusEpochs/correct vs error/%i_1.png' % (subject)) # session 1
        if session == 2:
            plt.savefig('Figures/TaskstimulusEpochs/correct vs error/%i_2.png' % (subject)) # session 2
        
        
        zscore = lambda x: (x - x.mean()) / x.std()
        task_stimulus_epochs.metadata['confidence_z_by_partner'] = task_stimulus_epochs.metadata['participant_confidence'].groupby(task_stimulus_epochs.metadata['partner']).transform(zscore)
        
        median_confidence = task_stimulus_epochs.metadata['confidence_z_by_partner'].median()
        low_conf_epochs = task_stimulus_epochs['confidence_z_by_partner <= %i' % median_confidence]
        high_conf_epochs = task_stimulus_epochs['confidence_z_by_partner > %i' % median_confidence]
        low_conf_evoked = low_conf_epochs.average()
        high_conf_evoked = high_conf_epochs.average()
        evokeds = dict(high_confidence=high_conf_evoked, low_confidence=low_conf_evoked)
        mne.viz.plot_compare_evokeds(evokeds, picks=['Pz'], invert_y=False)
        plt.title('stimulus-locked | P%i | %s' % (subject, condition))
        if session == 1:
            plt.savefig('Figures/TaskstimulusEpochs/z-scored confidence median split/%i_1.png' % (subject))
        if session == 2:
            plt.savefig('Figures/TaskstimulusEpochs/z-scored confidence median split/%i_2.png' % (subject))

>>>> Session 1
>>>> Participant 1
Opening raw data file EEGdata/cleaned/s1-raw.fif...
    Range : 0 ... 830749 =      0.000 ...  3322.996 secs
Ready.
Reading 0 ... 830749  =      0.000 ...  3322.996 secs...
Filtering raw data in 1 contiguous segment
Setting up low-pass filter at 8 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 8.00 Hz
- Upper transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 9.00 Hz)
- Filter length: 413 samples (1.652 sec)

2844 events found
Event IDs: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]
Not setting metadata
Not setting metadata
405 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Loading data for 405 events and 326 original time points ...
1 bad epochs dropped
Adding metadata with 50 columns
    Rejecting

Setting up low-pass filter at 8 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 8.00 Hz
- Upper transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 9.00 Hz)
- Filter length: 413 samples (1.652 sec)

2844 events found
Event IDs: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]
Not setting metadata
Not setting metadata
405 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Loading data for 405 events and 326 original time points ...
13 bad epochs dropped
Adding metadata with 50 columns
    Rejecting  epoch based on EEG : ['Fp1']
    Rejecting  epoch based on EEG : ['AF3']
    Rejecting  epoch based on EEG : ['AF3']
    Rejecting  epoch based on EEG : ['FCz']
    Rejecting  epoch based on EEG : ['AF3']
    Rejecting  epoch based on EEG : ['AF4'

c:\users\majaf\miniconda3\lib\site-packages\mne\viz\utils.py:1214: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  fig = plt.figure(figsize=figsize)


Not setting metadata
Not setting metadata
280 matching events found
No baseline correction applied
0 projection items activated
0 bad epochs dropped
>>>> Session 1
>>>> Participant 9
Opening raw data file EEGdata/cleaned/s9-raw.fif...
    Range : 0 ... 853999 =      0.000 ...  3415.996 secs
Ready.
Reading 0 ... 853999  =      0.000 ...  3415.996 secs...
Filtering raw data in 1 contiguous segment
Setting up low-pass filter at 8 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 8.00 Hz
- Upper transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 9.00 Hz)
- Filter length: 413 samples (1.652 sec)

Trigger channel has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2844 events found
Event IDs

Ready.
Reading 0 ... 989749  =      0.000 ...  3958.996 secs...
Filtering raw data in 1 contiguous segment
Setting up low-pass filter at 8 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 8.00 Hz
- Upper transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 9.00 Hz)
- Filter length: 413 samples (1.652 sec)

Trigger channel has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2843 events found
Event IDs: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14]
Not setting metadata
Not setting metadata
405 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Loading data for 405 events and 326 original time points ...
0 bad epochs dropped
Adding metadata with 50 c

Removing orphaned offset at the beginning of the file.
2238 events found
Event IDs: [ 1  2  3  4  5  6  7  8  9 10 11]
Not setting metadata
Not setting metadata
405 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Loading data for 405 events and 326 original time points ...
0 bad epochs dropped
Adding metadata with 50 columns
    Rejecting  epoch based on EEG : ['Iz']
    Rejecting  epoch based on EEG : ['Oz']
    Rejecting  epoch based on EEG : ['Oz']
    Rejecting  epoch based on EEG : ['Oz']
4 bad epochs dropped
Overwriting existing file.
No projector specified for this dataset. Please consider the method self.add_proj.
Not setting metadata
Not setting metadata
296 matching events found
No baseline correction applied
0 projection items activated
0 bad epochs dropped
>>>> Session 1
>>>> Participant 21
Opening raw data file EEGdata/cleaned/s21-raw.fif...
    Range : 0 ... 748249 =      0.000 ...  2992.996 secs
Ready.
Reading 0 ... 748249  = 

2844 events found
Event IDs: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]
Not setting metadata
Not setting metadata
405 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Loading data for 405 events and 326 original time points ...
0 bad epochs dropped
Adding metadata with 50 columns
0 bad epochs dropped
Overwriting existing file.
No projector specified for this dataset. Please consider the method self.add_proj.
Not setting metadata
Not setting metadata
300 matching events found
No baseline correction applied
0 projection items activated
0 bad epochs dropped
>>>> Session 2
>>>> Participant 6
Opening raw data file EEGdata/cleaned/s6_2-raw.fif...
    Range : 0 ... 812999 =      0.000 ...  3251.996 secs
Ready.
Reading 0 ... 812999  =      0.000 ...  3251.996 secs...
Filtering raw data in 1 contiguous segment
Setting up low-pass filter at 8 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter

Overwriting existing file.
No projector specified for this dataset. Please consider the method self.add_proj.
Not setting metadata
Not setting metadata
299 matching events found
No baseline correction applied
0 projection items activated
0 bad epochs dropped
>>>> Session 2
>>>> Participant 12
Opening raw data file EEGdata/cleaned/s12_2-raw.fif...
    Range : 0 ... 903749 =      0.000 ...  3614.996 secs
Ready.
Reading 0 ... 903749  =      0.000 ...  3614.996 secs...
Filtering raw data in 1 contiguous segment
Setting up low-pass filter at 8 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 8.00 Hz
- Upper transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 9.00 Hz)
- Filter length: 413 samples (1.652 sec)

2843 events found
Event IDs: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14]
Not se

- Upper passband edge: 8.00 Hz
- Upper transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 9.00 Hz)
- Filter length: 413 samples (1.652 sec)

Trigger channel has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2239 events found
Event IDs: [ 1  2  3  4  5  6  7  8  9 10 11 15]
Not setting metadata
Not setting metadata
405 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Loading data for 405 events and 326 original time points ...
0 bad epochs dropped
Adding metadata with 50 columns
0 bad epochs dropped
Overwriting existing file.
No projector specified for this dataset. Please consider the method self.add_proj.
Not setting metadata
Not setting metadata
300 matching events found
No baseline correction applied
0 projection items activated
0 bad epochs dropped
>>>> Session 2
>>>> Participant 18
Opening raw data file EEGdata/cleaned/s18_2-raw.fif...